In [1]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
from src.experiment import build_exp_graph

In [10]:
def load_design(subj):
    inFolder = 'C:\\Users\\Zhou Xiaoyue\\OneDrive - University College London\\Guenevere\\UCL\\Ph.D.project\\code.exp\\design\\'
    filename = inFolder + 'des_{}_subj.mat'.format(subj)
    des = loadmat(filename)
    
    expo_seq = des['dm_expo']['stim_seq'][0,0]
    trials_s1 = des['dm_2AFC']['seq_mat'][0,0]
    trials_s2 = des['dm_2AFC_conso']['seq_mat'][0,0]
    
    return expo_seq-1, trials_s1-1, trials_s2-1  # adjust index difference

def load_result(subj):
    inFolder = 'C:\\Users\\Zhou Xiaoyue\\OneDrive - University College London\\Guenevere\\UCL\\Ph.D.project\\code.exp\\data_batch2\\'

    # session 1
    filename = inFolder + 'ForcedChoice_{}_subj.mat'.format(subj)
    dat = loadmat(filename)

    rt = dat['result_2AFC']['rt'][0,0]
    answ = dat['result_2AFC']['answ'][0,0]
    resp = dat['result_2AFC']['resp'][0,0]

    result_s1 = pd.DataFrame({
        'rt': rt.ravel().astype(float),
        'answ': -1 * answ.ravel().astype(int) + 2,
        'resp': -1 * resp.ravel().astype(int) + 2
    })

    # session 2
    filename = inFolder + 'ForcedChoice_{}_subj_conso.mat'.format(subj)
    dat = loadmat(filename)

    rt = dat['result_2AFC']['rt'][0,0]
    answ = dat['result_2AFC']['answ'][0,0]
    resp = dat['result_2AFC']['resp'][0,0]

    result_s2 = pd.DataFrame({
        'rt': rt.ravel().astype(float),
        'answ': -1 * answ.ravel().astype(int) + 2,
        'resp': -1 * resp.ravel().astype(int) + 2
    })

    result_s1['session'] = 'S1'
    result_s1['trial'] = np.arange(0, 400).astype(int)
    result_s2['session'] = 'S2'
    result_s2['trial'] = np.arange(0, 400).astype(int)

    return pd.concat([result_s1, result_s2])

def seqMat_to_trials(seq_mat, adjacency, community):
    n_trial = seq_mat.shape[0]
    trial = np.arange(0,400)
    block = trial // 80
    cue = seq_mat[:,-2].astype(int)
    target = seq_mat[:,-1].astype(int)

    legal = adjacency[cue, target].astype(bool)
    within = community[cue] == community[target]
    sequence = seq_mat.astype(int)
    sequence = [row.tolist() for row in sequence]

    membership_lab = np.where(within, 'within', 'between')
    legal_lab = np.where(legal, 'legal', 'illegal')
    trial_type = np.char.add(membership_lab, '_')
    trial_type = np.char.add(trial_type, legal_lab)

    df = pd.DataFrame({
        'trial': trial,
        'trial_in_block': np.tile(np.arange(80), 5),
        'block': block,
        'cue': cue,
        'target': target,
        'legal': legal,
        'within': within,
        'sequence': sequence,
        'trial_type': trial_type
    })
    return df

def get_full_df(session):
    adjacency, community, _ = build_exp_graph()
    subj_tbl = []
    for subj in subj_list:
        _, trials_s1, trials_s2 = load_design(subj)
        if session == 'S1':
            trials = seqMat_to_trials(
                trials_s1,
                adjacency=adjacency,
                community=community
            )
        elif session == 'S2':
            trials = seqMat_to_trials(
                trials_s2,
                adjacency=adjacency,
                community=community
            )
        else:
            raise(ValueError('Wrong session string - should be either S1 or S2.'))

        result = load_result(subj)
        result = result[result['session'] == session]

        grand = pd.concat([trials, result], axis=1)
        # remove time-out trials
        grand = grand.loc[grand['resp'].isin([0,1])].copy()
        grand['subj'] = str(subj)
        subj_tbl.append(grand)
    
    output = pd.concat(subj_tbl, ignore_index=True)
    return output

def bootstrap(df_full, sampled_subj):    
    sampled_df = []
    for iSubj, subj in enumerate(sampled_subj):
        df_subj = df_full[df_full['subj'] == str(subj)].copy()
        df_subj['original_subj'] = df_subj['subj']
        df_subj['subj'] = str(iSubj)

        sampled_df.append(df_subj)
    
    return pd.concat(sampled_df, ignore_index=True)


In [23]:
n_iter = 10
session = 'S1'

subj_full = list(range(121, 155))
idx_remov = [128, 134, 140, 144, 152]
subj_list = [x for x in subj_full if x not in idx_remov]

adjacency, community, node_degree = build_exp_graph()
n_node = adjacency.shape[0]

rng = np.random.default_rng(123)

sampled_subj = rng.choice(subj_list,
                            len(subj_list),
                            replace=True)
df_full = get_full_df(session)

sampled_df = bootstrap(df_full, sampled_subj)

# get group-level matrix of p_yes
mat = (
    sampled_df
    .groupby(['subj', 'cue', 'target'])
    .agg(p = ('resp', 'mean'))
    .groupby(['cue', 'target'])
    .agg(p_mean = ('p', 'mean'))
)

mat = mat['p_mean'].unstack('target')

n_within = 0
n_between = 0
val_within = 0
val_between = 0
for i in range(n_node):
    for j in range(n_node):
        if i == j:
            continue
        
        is_same_community = community[i] == community[j]
        p_val = mat.loc[i, j] # check here
        if is_same_community:
            n_within += 1
            val_within += p_val
        else:
            n_between += 1
            val_between += p_val
dist_within = val_within / n_within
dist_between = val_between / n_between

print(f'within-community distance is {dist_within}, \n between-community distance is {dist_between}')

within-community distance is 0.5898965517241379, 
 between-community distance is 0.47586786833146977


In [ ]:
for iter in range(n_iter):
    sampled_subj = rng.choice(subj_list,
                              len(subj_list),
                              replace=True)
    df_full = get_full_df(session)
    sampled_df = bootstrap(df_full, sampled_subj)

    # get group-level matrix of p_yes
    mat = (
        sampled_df
        .groupby(['subj', 'cue', 'probe'])
        .agg(p = ('resp', 'mean'))
        .groupby(['cue', 'probe'])
        .agg(p_mean = ('p', 'mean'))
    )

    # calculate meansurement
    # [1.1] distance of within-community pairs, 
    # [1.2] distance of between-community pairs
    n_within = 0
    n_between = 0
    val_within = 0
    val_between = 0
    for i in range(n_node):
        for j in range(n_node):
            if i == j:
                continue
            
            is_same_community = community[i] == community[j]
            p_val = mat[mat['cue'] == i and mat['probe'] == j] # check here
            if is_same_community:
                n_within += 1
                val_within += p_val
            else:
                n_between += 1
                val_between += p_val
    dist_within = val_within / n_within
    dist_between = val_between / n_between
    # record results here

    # [2] community seperation for each node
    dist_same = np.zeros(n_node, 1)
    dist_other = np.zeros(n_node, 1)
    for i in range(n_node):
        cur_dist_same = 0
        cur_dist_other = 0
        n_same = 0
        n_other = 0
        for j in range(n_node):
            if i==j:
                continue
            is_same_community = community[i] == community[j]
            p_val = mat[mat['cue'] == i and mat['probe'] == j] # check here
            if is_same_community:
                n_same += 1
                cur_dist_same += p_val
            else:
                n_other += 1
                cur_dist_other += p_val
        dist_same[i] = cur_dist_same / n_same
